# Setup y carga de datos

In [ ]:
import os
from google.colab import userdata

In [ ]:
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!pip install -q kaggle==1.6.17

In [ ]:
!kaggle datasets download -d rusiano/madrid-airbnb-data
!unzip -o madrid-airbnb-data.zip -d data/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")

In [ ]:
print("Archivos disponibles: ", os.listdir("data/"))

In [ ]:
df = pd.read_csv("data/listings.csv")
print(f"Shape: {df.shape}")
df.head()

# Inspeccion inicial

In [ ]:
df.info()

display(df.describe())

nulls = (df.isnull().sum() / len(df)*100).round(2)
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(nulls)

# Limpieza de datos

In [ ]:
# Eliminar los price = 0 (si lo hay)
cantidad_ceros = (df["price"]== 0).sum()
print(cantidad_ceros)

df = df[df["price"] > 0].copy() # El nuevo df solo coge de la columna price los valores mayores a 0

col_analisis = df[["id", "name", "host_id", "host_name", "neighbourhood_group", "neighbourhood", "latitude", "longitude", "room_type", "price",
                "minimum_nights", "number_of_reviews", "last_review","reviews_per_month", "calculated_host_listings_count","availability_365"]]

df_clean = col_analisis.copy()

# Rellenar nulos

#reviews_per_month
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

#last_review
df_clean["last_review"] = df_clean["last_review"].fillna("NaT")
df_clean["last_review"] = pd.to_datetime(df_clean["last_review"])

#host_name
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown")

#name
df_clean["name"] = df_clean["name"].fillna("Unknown")

# EDA - Análisis Univariante

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(15,15))

price_cap = df_clean["price"].quantile(0.99)

# Distribucion de precios

ax = axes[0,0]
df_clean[df_clean["price"] <= price_cap]["price"].hist(bins=50, ax=ax, color="#ff5a5f", edgecolor= "white")

ax.set_title("Distribución de Precios")

ax.set_xlabel("Precio (€)")
ax.set_ylabel("Frecuencia")
ax.axvline(df_clean["price"].median(), color="#00a699", linestyle="-", label=f"Mediana: {df_clean["price"].median():.0f}€")

# Listings por Tipo de Habitación

ax = axes[0,1]
room_counts = df_clean["room_type"].value_counts()
room_counts.plot(kind="bar", ax=ax, color="#ff5a5f")

ax.set_title("Listings por Tipo de Habitación")

ax.set_ylabel("Cantidad")
ax.tick_params(axis="x", rotation=45)

# Top 20 Barrios por Listings

ax = axes[1,0]
top_barrios = df_clean["neighbourhood"].value_counts().head(20)
top_barrios.plot(kind="barh", ax=ax, color="#ff5a5f")

ax.set_title("Top 20 Barrios por Listings")

ax.set_ylabel = ("Cantidad")
ax.invert_yaxis()

# Distribución Mínima de Noches

ax = axes[1,1]
min_night_cap = df_clean["minimum_nights"].clip(upper=30)
min_night_cap.hist(bins=30, ax=ax, color="#ff5a5f", edgecolor = "white")

ax.set_title("Distribución Mínimo de Noches")

ax.set_xlabel = ("Noches Mínimas")

plt.tight_layout()
plt.savefig("01_Analisis_Univariante.png", dpi=150, bbox_inches = "tight")
plt.show()


# EDA - Análisis Bivariante y Geográfico

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))

price_cap = df_clean["price"].quantile(0.99)
df_plot = df_clean[df_clean["price"] <= price_cap]

# Precio por habitación
ax= axes[0]

sns.boxplot(data=df_plot, x="room_type", y="price", ax=ax, palette="Set2")

ax.set_title("Precio por Tipo de Habitación")

ax.set_ylabel("Precio (€)")

# Precio por barrio (Top 20)

ax= axes[1]

precio_barrio = (df_clean.groupby("neighbourhood")["price"]).agg(["median", "count"]).query("count >= 50").sort_values("median", ascending=False).head(20)
precio_barrio["median"].plot(kind="barh", ax=ax, color="#ff5a5f")

ax.set_title("Top 20 Barrios Más Caros (min. 50 listings)")

ax.set_xlabel("Precio Mediano (€)")
ax.invert_yaxis()


plt.tight_layout()
plt.savefig("02_Analisis_Bivariante.png", dpi=150, bbox_inches = "tight")
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

fig, ax = plt.subplots(figsize=(10,10))
ax.set_facecolor("#1a1a2e")
fig.set_facecolor("#1a1a2e")


airbnb_cmap = LinearSegmentedColormap.from_list("airbnb", ["#FFFFFF", "#FFB7B2", "#FF5A5F", "#CC0000"])


price_cap = df_clean["price"].quantile(0.99)
df_plot = df_clean[df_clean["price"] <= price_cap]

scatter = ax.scatter(df_plot["longitude"], df_plot["latitude"], c=df_plot["price"], cmap=airbnb_cmap, alpha=0.6, s=10, edgecolors="none")

cbar = plt.colorbar(scatter, label="Precio (€)")
cbar.ax.yaxis.set_tick_params(color="white")
cbar.ax.yaxis.label.set_color("white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

ax.set_title("Mapa de Listings en Madrid — Precio por ubicación", color="white", fontsize=14)

ax.set_xlabel("Longitud", color="white")
ax.set_ylabel("Latitud", color="white")
ax.tick_params(colors="white")
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig("03_mapa_geografico.png", dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

# Matriz de correlación

In [ ]:
cols_num = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cols_num = [c for c in cols_num
            if c not in ["id", "host_id"]]

corr_matrix = df_clean[cols_num].corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap=airbnb_cmap, center=0, ax=ax, square=True, linewidths=0.5)

ax.set_title("Matriz de Correlaciones")

plt.tight_layout()
plt.savefig("04_matriz_de_correlaciones.png", dpi=150, bbox_inches="tight")
plt.show()

print("Correlación con Price")
print(corr_matrix["price"].drop("price").sort_values(ascending=False))

# Predicción de precios (Machine Learning)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df_ml = df_clean[["latitude", "longitude", "room_type", "price","minimum_nights", "number_of_reviews", "reviews_per_month", "calculated_host_listings_count", "availability_365", "neighbourhood"]].copy()

df_ml = df_ml[df_ml["price"] <= df_ml["price"].quantile(0.99)]
df_ml = df_ml.dropna()

le_room = LabelEncoder()
df_ml["room_type_enc"] = le_room.fit_transform(df_ml["room_type"])
print("Encoding room_type:", dict(zip(le_room.classes_, le_room.transform(le_room.classes_))))


features = ["latitude", "longitude","room_type_enc", "minimum_nights", "number_of_reviews", "reviews_per_month", "calculated_host_listings_count", ]

x = df_ml[features + ["neighbourhood"]].dropna()
y = df_ml.loc[x.index, "price"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

barrio_freq = x_train["neighbourhood"].value_counts(normalize=True)
barrio_median = df_ml.groupby("neighbourhood")["price"].median()

x_train["neighbourhood_freq"] = x_train["neighbourhood"].map(barrio_freq).fillna(0)

x_test["neighbourhood_freq"] = x_test["neighbourhood"].map(barrio_freq).fillna(0)

x_train = x_train.drop(columns=["neighbourhood"])
x_test = x_test.drop(columns=["neighbourhood"])

features = x_train.columns.tolist()
print(f"Features: {features}")
print(f"Train: {len(x_train)} | Test: {len(x_test)}")

models = {"Regresión Lineal": LinearRegression(), "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=1), "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state= 42)}

resultados = []

for name, models in models.items():
    models.fit(x_train, y_train)
    y_pred = models.predict(x_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados.append({"Modelo": name, "MAE": round(mae,2), "RMSE": round(rmse,2), "R2": round(r2,2)})

    print(f"Modelo: {name}")
    print(f"MAE: {mae:2f} €")
    print(f"RMSE: {rmse:2f} €")
    print(f"R2: {r2*100:2f} %")

df_resultados = pd.DataFrame(resultados)
display(df_resultados)


# Feature importance + prediccion vs real

In [ ]:
best_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=1)

best_model.fit(x_train, y_train)
y_pred_best = best_model.predict(x_test)

fig, axes =plt.subplots(1, 2, figsize=(14,6))

# Importancia

ax = axes[0]

importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=x_train.columns).sort_values(ascending=True)
feat_imp.plot(kind="barh", ax=ax, color="#ff5a5f")

ax.set_title("Feature Importance")

ax.set_xlabel("Importancia")

# Predicción vs real

ax = axes[1]

ax.scatter(y_test, y_pred_best, alpha=0.3, s=10, color="#ff5a5f")
max_val = max(y_test.max(), y_pred_best.max())

ax.plot([0, max_val], [0, max_val], "k--", linewidth=2, label= "Predcción perfecta")

ax.set_title("Predicción vs Precio Real")

ax.set_xlabel("Precio Real (€)")
ax.set_ylabel("Predicción")

ax.legend()

plt.tight_layout()
plt.savefig("05_modelo_resultados.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
residuos = y_test - y_pred_best

fig, ax = plt.subplots(figsize=(10,4))

ax.hist(residuos, bins=50, color="#ff5a5f", edgecolor="white")
ax.axvline(0, color="black", linestyle="--", linewidth=2)

ax.set_title("Distribución de Residuos")

ax.set_xlabel("Error")
ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig("06_residuos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_clean.to_csv("madrid_airbnb.csv", index=False)

from google.colab import files
files.download("madrid_airbnb.csv")